In [ ]:
!pip3 install --upgrade tensorflow_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import cv2

import PIL.Image as Image
import os

import matplotlib.pylab as plt

import tensorflow as tf
import tensorflow_hub as hub

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [ ]:
# Get this model from tensorflow hub
#IMAGE_SHAPE = (224, 224)
# classifier = tf.keras.Sequential([
#     hub.KerasLayer("https://tfhub.dev/google/tf2-preview/mobilenet_v2/classification/4", input_shape=IMAGE_SHAPE+(3,))
# ])

In [ ]:
import tensorflow_hub as hub
import tensorflow as tf

# This is a TF-Hub "handle" (URL) for MobileNetV2
# Breakdown:
# - mobilenet_v2      : model architecture
# - 035               : width multiplier = small & fast model
# - 224               : trained on 224x224 images
# - classification    : includes final softmax layer for ImageNet
# - /5                : version number
model_handle = "https://tfhub.dev/google/imagenet/mobilenet_v2_035_224/classification/5"

#Load the pretrained model
# ----------------------------------------------------------

# hub.KerasLayer loads the pretrained model as a reusable Keras layer.
# This gives us a fully trained ImageNet classifier.
# No training required — we can directly pass 224x224x3 images to it.
model = hub.KerasLayer(model_handle)



In [ ]:
# import tensorflow_hub as hub
# m = tf.keras.Sequential([
#     hub.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-224-classification/2")
# ])
# m.build([None, 224, 224, 3])  # Batch input shape.


In [ ]:
# Create a Sequential model
# We are using Lambda layer to wrap a TF-Hub pretrained model
from tensorflow.keras.layers import Lambda
m = tf.keras.Sequential([

    # Lambda allows custom operations inside the layer.
    # Here, we load a MobileNetV2 pretrained model from Kaggle TFHub.
    # The model accepts images of size 224x224x3.
    Lambda(lambda x: hub.KerasLayer(
        "https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-224-classification/2"
    )(x))
])

# Build the model by defining the expected input shape
# None = batch size (variable)
# 224,224,3 = image height, width, channels
m.build([None, 224, 224, 3])

# Print model summary to verify layer structure (optional)

In [ ]:
m.summary()

In [ ]:
tf.keras.utils.get_file(fname='ImageNetLabels.txt', # Changed fname to just the filename
                        origin='https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt',
                        cache_dir='/content/')

In [ ]:
with open("/content/datasets/ImageNetLabels.txt", "r") as f:
    image_labels = f.read().splitlines()

In [ ]:
print(image_labels)


Double-click (or enter) to edit

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
mypic = cv2.imread("/content/images.webp")
cv2_imshow(mypic)

In [ ]:
# Print the original shape of the image
print(mypic.shape)

# ------------------------------------------------------------
# 1. Resize the image → MobilenetV2 expects 224x224x3 input
# ------------------------------------------------------------
resized_pic = cv2.resize(mypic, (224, 224))
print(resized_pic.shape)        # Should print (224, 224, 3)

# ------------------------------------------------------------
# 2. Normalize pixel values → converts [0,255] to [0,1]
# ------------------------------------------------------------
resized_pic = resized_pic / 255.0

# ------------------------------------------------------------
# 3. Create a batch → Model expects batch dimension (N, 224, 224, 3)
# Here we only have ONE image → so batch size = 1
# ------------------------------------------------------------
batch = resized_pic.reshape(1, 224, 224, 3)

# ------------------------------------------------------------
# 4. Make prediction using the loaded model `m`
# Prediction output shape → (1, 1001) for ImageNet
# ------------------------------------------------------------
output = m.predict(batch)

# ------------------------------------------------------------
# 5. Get index of class with highest probability
# ------------------------------------------------------------
output_ind = np.argmax(output)

# ------------------------------------------------------------
# 6. Fetch the label name using the index from your labels list
# ------------------------------------------------------------
predicted_label = image_labels[output_ind]
print("Predicted Label:", predicted_label)


In [ ]:
image_labels[np.argmax(m.predict(batch))]


In [ ]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file(fname='flower_photos', origin=dataset_url,  cache_dir='.', untar=True)
# cache_dir indicates where to download data. I specified . which means current directory
# untar true will unzip it

In [ ]:
data_dir = "/content/datasets/flower_photos/flower_photos"

In [ ]:
type(data_dir)

In [ ]:
# Import pathlib to work with filesystem paths in an easy, OS-independent way
import pathlib

# 'data_dir' at this point is most likely a STRING returned by:
# tf.keras.utils.get_file(..., untar=True)
# Example: data_dir = "./flower_photos"
# We now convert this string into a proper Path object.

data_dir = pathlib.Path(data_dir)

# Print the type of data_dir to verify it is now a pathlib Path object.
# This will show: <class 'pathlib.PosixPath'>
type(data_dir)


In [ ]:
data_dir.glob("*") ## gives generator object

In [ ]:
list(data_dir.glob("*"))

In [ ]:
list(data_dir.glob("tulips/*"))

In [ ]:
# Get all image paths inside the 'roses' folder
rose_paths = list((data_dir / "roses").glob("*"))
rose_paths[:5]


In [ ]:
len(list(data_dir.glob("roses/*")))

In [ ]:
print(list(data_dir.glob("roses/*")))

In [ ]:
list(data_dir.glob("roses/*"))[1]

In [ ]:
from google.colab.patches import cv2_imshow
myimg = cv2_imshow(cv2.imread(list(data_dir.glob("tulips/*"))[10]))

In [ ]:
print(len(list(data_dir.glob("roses/*"))))
print(len(list(data_dir.glob("daisy/*"))))
print(len(list(data_dir.glob("dandelion/*"))))
print(len(list(data_dir.glob("sunflowers/*"))))
print(len(list(data_dir.glob("tulips/*"))))

In [ ]:
641 + 633+898+699+799

In [ ]:
flowers_images_dict = {
    'roses': list(data_dir.glob('roses/*')),
    'daisy': list(data_dir.glob('daisy/*')),
    'dandelion': list(data_dir.glob('dandelion/*')),
    'sunflowers': list(data_dir.glob('sunflowers/*')),
    'tulips': list(data_dir.glob('tulips/*')),
}
flowers_labels_dict = {
    'roses': 0,
    'daisy': 1,
    'dandelion': 2,
    'sunflowers': 3,
    'tulips': 4,
}

In [ ]:
X, y = [], [] # empty list

for flower_name, images in flowers_images_dict.items():
    for image in images:
        img = cv2.imread(image)
        resized_img = cv2.resize(img,(224,224))
        resized_img = resized_img.astype("float32")/255
        X.append(resized_img)
        y.append(flowers_labels_dict[flower_name])

In [ ]:
len(X)

In [ ]:
from google.colab.patches import cv2_imshow
cv2_imshow(X[34])

In [ ]:
y[34]

In [ ]:
X = np.array(X)
y = np.array(y)

In [ ]:
X.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.1 ,stratify=y ,random_state=0)

In [ ]:
# feature_extractor_model = "https://tfhub.dev/google/tf2-preview/mobilenet_v2/feature_vector/4"

# pretrained_model_without_top_layer = hub.KerasLayer(
#     feature_extractor_model, input_shape=(224, 224, 3), trainable=False)
# pretrained_model_without_top_layer

In [ ]:
# import tensorflow_hub as hub
# num_classes = 5
# m = tf.keras.Sequential([
#     hub.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-224-feature-vector/2",
#                    trainable=False),  # Can be True, see below.
#     tf.keras.layers.Dense(num_classes, activation='softmax')
# ])
# m.build([None, 224, 224, 3])  # Batch input shape.


In [ ]:
model = tf.keras.Sequential([
  Lambda(lambda x : hub.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-224-feature-vector/2")(x)),
  keras.layers.Flatten(),
  keras.layers.Dense(200, activation='relu'),
  keras.layers.Dense(5,activation='softmax')
])
model.build([None,224,224,3])
model.summary()

In [ ]:
model.compile(
  optimizer="adam",
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['acc'])

model.fit(X_train,y_train, epochs=5)

In [ ]:
# m.compile(
#   optimizer="adam",
#   loss=tf.keras.losses.SparseCategoricalCrossentropy(),
#   metrics=['acc'])

# m.fit(X_train, y_train, epochs=5)

In [ ]:
demopic = cv2.imread("/content/datasets/flower_photos/flower_photos/roses/12202373204_34fb07205b.jpg")
demopic = cv2.resize(demopic,(224,224)) # resizing
demopic = demopic.astype("float32")/255 # rescaling from 0-255 to 0-1
batch = demopic.reshape(1,224,224,3) # reshape --> (no. of images, rows, cols, channels)
output = model.predict(batch)
print(output)# print probability values for all categories
print(output.max()) # print max probability value
print(np.argmax(output)) # fetching index of max probability value getting from o/p layer neuron (getting this in the form of vectors of probability values for all categories)
